## 🎯 PARTIE 7: APPLICATION PRATIQUE - DATASET IRIS

### Cas d'Usage Réel

Appliquons nos techniques de clustering sur le célèbre dataset Iris :
- **3 espèces** de fleurs d'iris
- **4 caractéristiques** : longueur/largeur des sépales et pétales
- **Objectif** : Retrouver les 3 espèces sans connaître les étiquettes

In [ ]:
# Chargement et exploration du dataset Iris
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print("Dataset Iris - Exploration:")
print(f"Forme des données: {X_iris.shape}")
print(f"Caractéristiques: {feature_names}")
print(f"Classes: {target_names}")
print(f"Distribution des classes: {np.bincount(y_iris)}")

# Visualisation des données originales
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

# Paires de caractéristiques
feature_pairs = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]

for i, (f1, f2) in enumerate(feature_pairs):
    ax = axes[i]
    scatter = ax.scatter(X_iris[:, f1], X_iris[:, f2], c=y_iris, cmap='viridis', alpha=0.7)
    ax.set_xlabel(feature_names[f1])
    ax.set_ylabel(feature_names[f2])
    ax.set_title(f'{feature_names[f1]} vs {feature_names[f2]}')
    ax.grid(True, alpha=0.3)

plt.colorbar(scatter, ax=axes[-1])
plt.tight_layout()
plt.show()

### Standardisation des Données

**Pourquoi standardiser ?**
- Les caractéristiques ont des échelles différentes
- K-means est sensible aux échelles
- La standardisation améliore les performances

In [ ]:
# Standardisation des données
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

print("Comparaison avant/après standardisation:")
print("\nAvant standardisation:")
print(f"Moyennes: {X_iris.mean(axis=0)}")
print(f"Écarts-types: {X_iris.std(axis=0)}")

print("\nAprès standardisation:")
print(f"Moyennes: {X_iris_scaled.mean(axis=0)}")
print(f"Écarts-types: {X_iris_scaled.std(axis=0)}")

# Application des algorithmes de clustering
print("\n" + "="*50)
print("APPLICATION DES ALGORITHMES DE CLUSTERING")
print("="*50)

# K-means
kmeans_iris = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_kmeans_iris = kmeans_iris.fit_predict(X_iris_scaled)

# Clustering hiérarchique
hierarchical_iris = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_hierarchical_iris = hierarchical_iris.fit_predict(X_iris_scaled)

# Évaluation des résultats
sil_kmeans_iris = silhouette_score(X_iris_scaled, labels_kmeans_iris)
sil_hierarchical_iris = silhouette_score(X_iris_scaled, labels_hierarchical_iris)

# Comparaison avec les vraies étiquettes (pour information)
ari_kmeans = adjusted_rand_score(y_iris, labels_kmeans_iris)
ari_hierarchical = adjusted_rand_score(y_iris, labels_hierarchical_iris)

print(f"\nRésultats sur Iris:")
print(f"K-means - Silhouette: {sil_kmeans_iris:.3f}, ARI: {ari_kmeans:.3f}")
print(f"Hiérarchique - Silhouette: {sil_hierarchical_iris:.3f}, ARI: {ari_hierarchical:.3f}")

### Visualisation des Résultats

In [ ]:
# Visualisation comparative des résultats
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Données originales (vraies classes)
axes[0,0].scatter(X_iris_scaled[:, 0], X_iris_scaled[:, 1], c=y_iris, cmap='viridis', alpha=0.7)
axes[0,0].set_title('Vraies Classes')
axes[0,0].set_xlabel(feature_names[0])
axes[0,0].set_ylabel(feature_names[1])

# K-means
axes[0,1].scatter(X_iris_scaled[:, 0], X_iris_scaled[:, 1], c=labels_kmeans_iris, cmap='viridis', alpha=0.7)
axes[0,1].scatter(kmeans_iris.cluster_centers_[:, 0], kmeans_iris.cluster_centers_[:, 1], 
                  c='red', marker='x', s=200, linewidths=3)
axes[0,1].set_title(f'K-means (Silhouette: {sil_kmeans_iris:.3f})')
axes[0,1].set_xlabel(feature_names[0])
axes[0,1].set_ylabel(feature_names[1])

# Clustering hiérarchique
axes[1,0].scatter(X_iris_scaled[:, 0], X_iris_scaled[:, 1], c=labels_hierarchical_iris, cmap='viridis', alpha=0.7)
axes[1,0].set_title(f'Hiérarchique (Silhouette: {sil_hierarchical_iris:.3f})')
axes[1,0].set_xlabel(feature_names[0])
axes[1,0].set_ylabel(feature_names[1])

# Dendrogramme pour le clustering hiérarchique
Z_iris = linkage(X_iris_scaled, method='ward')
dendrogram(Z_iris, ax=axes[1,1], truncate_mode='level', p=3)
axes[1,1].set_title('Dendrogramme - Dataset Iris')
axes[1,1].set_xlabel('Index des échantillons')
axes[1,1].set_ylabel('Distance')

plt.tight_layout()
plt.show()

## 🔧 PARTIE 8: TECHNIQUES AVANCÉES

### 1. Clustering avec Différentes Formes

K-means assume des clusters sphériques. Voyons ses limites :

In [ ]:
# Génération de données avec différentes formes
from sklearn.datasets import make_moons, make_circles

# Données en forme de lunes
X_moons, y_moons = make_moons(n_samples=200, noise=0.1, random_state=42)

# Données en forme de cercles
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.3, random_state=42)

# Application de K-means sur ces données
datasets = [
    (X_moons, y_moons, "Lunes"),
    (X_circles, y_circles, "Cercles")
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, (X, y_true, name) in enumerate(datasets):
    # Données originales
    axes[i, 0].scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', alpha=0.7)
    axes[i, 0].set_title(f'{name} - Vraies Classes')
    
    # K-means
    kmeans_shape = KMeans(n_clusters=2, random_state=42, n_init=10)
    labels_kmeans_shape = kmeans_shape.fit_predict(X)
    axes[i, 1].scatter(X[:, 0], X[:, 1], c=labels_kmeans_shape, cmap='viridis', alpha=0.7)
    axes[i, 1].scatter(kmeans_shape.cluster_centers_[:, 0], kmeans_shape.cluster_centers_[:, 1], 
                       c='red', marker='x', s=200, linewidths=3)
    axes[i, 1].set_title(f'{name} - K-means')
    
    # Clustering hiérarchique
    hierarchical_shape = AgglomerativeClustering(n_clusters=2, linkage='ward')
    labels_hierarchical_shape = hierarchical_shape.fit_predict(X)
    axes[i, 2].scatter(X[:, 0], X[:, 1], c=labels_hierarchical_shape, cmap='viridis', alpha=0.7)
    axes[i, 2].set_title(f'{name} - Hiérarchique')

plt.tight_layout()
plt.show()

print("Observation: K-means a des difficultés avec les formes non-sphériques")

### 2. Analyse de Sensibilité à l'Initialisation

In [ ]:
# Démonstration de la sensibilité de K-means à l'initialisation
def compare_initializations(X, n_clusters=3, n_runs=5):
    """Compare différentes initialisations de K-means"""
    
    fig, axes = plt.subplots(1, n_runs, figsize=(20, 4))
    
    silhouette_scores = []
    
    for i in range(n_runs):
        # K-means avec différentes graines aléatoires
        kmeans = KMeans(n_clusters=n_clusters, random_state=i, n_init=1)
        labels = kmeans.fit_predict(X)
        
        # Calcul du score de silhouette
        sil_score = silhouette_score(X, labels)
        silhouette_scores.append(sil_score)
        
        # Visualisation
        axes[i].scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', alpha=0.7)
        axes[i].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
                       c='red', marker='x', s=200, linewidths=3)
        axes[i].set_title(f'Init {i+1}\nSilhouette: {sil_score:.3f}')
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Scores de silhouette: {[f'{s:.3f}' for s in silhouette_scores]}")
    print(f"Écart-type: {np.std(silhouette_scores):.3f}")
    
    return silhouette_scores

print("Sensibilité à l'initialisation de K-means:")
scores = compare_initializations(X_demo, n_clusters=3, n_runs=5)

## 📈 PARTIE 9: MÉTRIQUES D'ÉVALUATION AVANCÉES

### Score de Silhouette Détaillé

Le score de silhouette pour chaque point mesure :
- **a** : Distance moyenne aux points du même cluster
- **b** : Distance moyenne aux points du cluster le plus proche
- **Silhouette** = (b - a) / max(a, b)

In [ ]:
from sklearn.metrics import silhouette_samples

# Calcul des scores de silhouette individuels
sample_silhouette_values = silhouette_samples(X_iris_scaled, labels_kmeans_iris)

# Visualisation des scores de silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Graphique de silhouette
y_lower = 10
colors = ['red', 'blue', 'green']

for i in range(3):
    # Scores pour le cluster i
    cluster_silhouette_values = sample_silhouette_values[labels_kmeans_iris == i]
    cluster_silhouette_values.sort()
    
    size_cluster_i = cluster_silhouette_values.shape[0]
    y_upper = y_lower + size_cluster_i
    
    ax1.fill_betweenx(np.arange(y_lower, y_upper),
                      0, cluster_silhouette_values,
                      facecolor=colors[i], edgecolor=colors[i], alpha=0.7)
    
    # Étiquette du cluster
    ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
    y_lower = y_upper + 10

ax1.set_xlabel('Score de Silhouette')
ax1.set_ylabel('Index des échantillons')
ax1.set_title('Graphique de Silhouette par Cluster')

# Ligne verticale pour le score moyen
ax1.axvline(x=sil_kmeans_iris, color="red", linestyle="--", 
            label=f'Score moyen: {sil_kmeans_iris:.3f}')
ax1.legend()

# Visualisation des clusters
ax2.scatter(X_iris_scaled[:, 0], X_iris_scaled[:, 1], c=labels_kmeans_iris, 
           cmap='viridis', alpha=0.7)
ax2.scatter(kmeans_iris.cluster_centers_[:, 0], kmeans_iris.cluster_centers_[:, 1], 
           c='red', marker='x', s=200, linewidths=3)
ax2.set_title('Clusters K-means')
ax2.set_xlabel(feature_names[0])
ax2.set_ylabel(feature_names[1])

plt.tight_layout()
plt.show()

## 🎯 PARTIE 10: CONSEILS PRATIQUES ET BONNES PRATIQUES

### Guide de Sélection d'Algorithme

| **Critère** | **K-means** | **Hiérarchique** |
|-------------|-------------|------------------|
| **Taille des données** | Grande (>1000) | Petite à moyenne (<1000) |
| **Forme des clusters** | Sphérique | Toute forme |
| **Nombre de clusters** | À définir à l'avance | Flexible |
| **Vitesse** | Rapide | Plus lent |
| **Déterminisme** | Non (initialisation) | Oui |
| **Interprétabilité** | Centroïdes | Dendrogramme |

### Workflow Recommandé

In [ ]:
def clustering_workflow(X, max_k=10, scale_data=True):
    """Workflow complet pour le clustering"""
    
    print("🔍 WORKFLOW DE CLUSTERING")
    print("="*50)
    
    # 1. Préparation des données
    if scale_data:
        scaler = StandardScaler()
        X_processed = scaler.fit_transform(X)
        print("✅ Données standardisées")
    else:
        X_processed = X
        print("ℹ️ Données non standardisées")
    
    # 2. Détermination du nombre optimal de clusters
    print("\n📊 Analyse du nombre optimal de clusters...")
    k_range, inertias, silhouette_scores = elbow_method(X_processed, max_k)
    
    # 3. Sélection du K optimal
    optimal_k_silhouette = k_range[np.argmax(silhouette_scores)]
    
    # Méthode du coude (approximation)
    # Calcul de la dérivée seconde pour trouver le coude
    if len(inertias) >= 3:
        second_derivatives = np.diff(inertias, 2)
        optimal_k_elbow = k_range[np.argmax(second_derivatives) + 2]
    else:
        optimal_k_elbow = 3
    
    print(f"K optimal (Silhouette): {optimal_k_silhouette}")
    print(f"K optimal (Coude): {optimal_k_elbow}")
    
    # 4. Application des algorithmes
    print(f"\n🤖 Application des algorithmes avec K={optimal_k_silhouette}...")
    
    # K-means
    kmeans = KMeans(n_clusters=optimal_k_silhouette, random_state=42, n_init=10)
    labels_kmeans = kmeans.fit_predict(X_processed)
    sil_kmeans = silhouette_score(X_processed, labels_kmeans)
    
    # Hiérarchique
    hierarchical = AgglomerativeClustering(n_clusters=optimal_k_silhouette, linkage='ward')
    labels_hierarchical = hierarchical.fit_predict(X_processed)
    sil_hierarchical = silhouette_score(X_processed, labels_hierarchical)
    
    # 5. Comparaison et recommandation
    print(f"\n📈 Résultats:")
    print(f"K-means - Silhouette: {sil_kmeans:.3f}")
    print(f"Hiérarchique - Silhouette: {sil_hierarchical:.3f}")
    
    if sil_kmeans > sil_hierarchical:
        print("🏆 Recommandation: K-means")
        best_labels = labels_kmeans
        best_algorithm = "K-means"
    else:
        print("🏆 Recommandation: Clustering Hiérarchique")
        best_labels = labels_hierarchical
        best_algorithm = "Hiérarchique"
    
    # 6. Visualisation finale
    if X_processed.shape[1] >= 2:
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        plt.plot(k_range, silhouette_scores, 'bo-')
        plt.axvline(x=optimal_k_silhouette, color='red', linestyle='--')
        plt.xlabel('Nombre de clusters (K)')
        plt.ylabel('Score de Silhouette')
        plt.title('Sélection du K optimal')
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 3, 2)
        plt.scatter(X_processed[:, 0], X_processed[:, 1], c=labels_kmeans, cmap='viridis', alpha=0.7)
        if hasattr(kmeans, 'cluster_centers_'):
            plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
                       c='red', marker='x', s=200, linewidths=3)
        plt.title(f'K-means (Sil: {sil_kmeans:.3f})')
        
        plt.subplot(1, 3, 3)
        plt.scatter(X_processed[:, 0], X_processed[:, 1], c=labels_hierarchical, cmap='viridis', alpha=0.7)
        plt.title(f'Hiérarchique (Sil: {sil_hierarchical:.3f})')
        
        plt.tight_layout()
        plt.show()
    
    return {
        'best_algorithm': best_algorithm,
        'best_labels': best_labels,
        'optimal_k': optimal_k_silhouette,
        'silhouette_scores': {'kmeans': sil_kmeans, 'hierarchical': sil_hierarchical}
    }

# Application du workflow sur le dataset Iris
print("Application du workflow sur le dataset Iris:")
results = clustering_workflow(X_iris, max_k=8, scale_data=True)

## 🎓 RÉSUMÉ ET CONCLUSIONS

### Ce que Nous Avons Appris

1. **Apprentissage Non Supervisé** : Découvrir des structures sans étiquettes
2. **Métriques de Distance** : Euclidienne, Manhattan, Minkowski
3. **Clustering Hiérarchique** : Construction d'une hiérarchie de clusters
4. **K-means** : Partitionnement en K clusters avec centroïdes
5. **Évaluation** : Silhouette, inertie, méthode du coude
6. **Applications Pratiques** : Workflow complet sur données réelles

### Points Clés à Retenir

✅ **Standardiser** les données avant clustering  
✅ **Tester plusieurs valeurs** de K  
✅ **Comparer différents algorithmes**  
✅ **Visualiser** les résultats  
✅ **Interpréter** les clusters obtenus  

### Prochaines Étapes

- **DBSCAN** : Clustering basé sur la densité
- **Gaussian Mixture Models** : Modèles de mélanges gaussiens
- **Clustering spectral** : Utilisation des valeurs propres
- **Applications avancées** : Segmentation d'images, analyse de texte

---

**Félicitations !** Vous maîtrisez maintenant les fondamentaux du clustering ! 🎉